In [5]:
using Pkg
Pkg.activate(".")
# Pkg.add(["DifferentialEquations", "ModelingToolkit", "StructuralIdentifiability"])
Pkg.status()
Pkg.add(["Makie", "AbstractPlotting"])

  Activating project at `c:\Users\MGAJ\OneDrive - Danmarks Tekniske Universitet\DTU\Kandidat\5_Semester\Speciale\discovering_hidden_physics\scripts\structural_identifiability`


Status `C:\Users\MGAJ\OneDrive - Danmarks Tekniske Universitet\DTU\Kandidat\5_Semester\Speciale\discovering_hidden_physics\scripts\structural_identifiability\Project.toml`
  [479239e8] Catalyst v15.0.8
  [0c46a032] DifferentialEquations v7.16.1
⌅ [961ee093] ModelingToolkit v9.82.0
  [295af30f] Revise v3.8.1
⌃ [220ca800] StructuralIdentifiability v0.5.14
Info Packages marked with ⌃ and ⌅ have new versions available. Those with ⌃ may be upgradable, but those with ⌅ are restricted by compatibility constraints from upgrading. To see why use `status --outdated`


    Updating registry at `C:\Users\MGAJ\.julia\registries\General.toml`
   Resolving package versions...
   Installed libfdk_aac_jll ───────────────── v2.0.4+0
   Installed TestSetExtensions ────────────── v2.0.0
   Installed ImageIO ──────────────────────── v0.4.1
   Installed x265_jll ─────────────────────── v4.1.0+0
   Installed TreeViews ────────────────────── v0.3.0
   Installed Antic_jll ────────────────────── v0.201.500+0
   Installed DifferentialEquations ────────── v7.6.0
   Installed Opus_jll ─────────────────────── v1.5.2+0
   Installed JuliaFormatter ───────────────── v1.0.62
   Installed Referenceables ───────────────── v0.1.3
   Installed FLINT_jll ────────────────────── v200.900.9+0
   Installed PNGFiles ─────────────────────── v0.3.17
   Installed StatsFuns ────────────────────── v0.9.18
   Installed Accessors ────────────────────── v0.1.27
   Installed RelocatableFolders ───────────── v0.1.3
   Installed Unitful ──────────────────────── v1.24.0
   Installed EarCut_jll 

In [2]:
using Revise, DifferentialEquations, ModelingToolkit, StructuralIdentifiability

In [3]:
## Lotka-Volterra equations
@parameters α β γ δ
@independent_variables  t
@variables x(t) y(t) z1(t) z2(t)
Dt = Differential(t)
eqs = [
    Dt(x) ~ α * x - β * x * y,
    Dt(y) ~ δ * x * y - γ * y,
]

params =  Dict([α => 0.1, 
                β => 0.02, 
                δ => 0.01,
                γ => 0.3])

u0 = Dict([x => 40.0, y => 9.0])

Dict{Num, Float64} with 2 entries:
  x(t) => 40.0
  y(t) => 9.0

# Setup 1. All states are observed

In [4]:
measured_quantities = [z1 ~ x, z2 ~ y]  # Observing both x and y
@named sys = ODESystem(eqs, t, [x, y], [α, β, γ, δ]; observed = measured_quantities)
sys = complete(sys)

Model sys:
Equations (2):
  2 standard: see equations(sys)
Unknowns (2): see unknowns(sys)
  x(t)
  y(t)
Parameters (4): see parameters(sys)
  α
  β
  δ
  γ
Observed (2): see observed(sys)

In [10]:
res = assess_identifiability(sys, measured_quantities = measured_quantities)

┌ Info: System parsed into x' = -x*y*β + x*α
│ y' = x*y*δ - y*γ
│ z1 = x
└ z2 = y
[ Info: Assessing local identifiability
[ Info: Assessing global identifiability
[ Info: Functions to check involve states
[ Info: Computing IO-equations
[ Info: Computed IO-equations in 7.8801646 seconds
[ Info: Computing Wronskians
[ Info: Computed Wronskians in 1.6542022 seconds
[ Info: Dimensions of the Wronskians [3, 3]
[ Info: Ranks of the Wronskians computed in 0.0137271 seconds
[ Info: Simplifying generating set. Simplification level: standard
✓ # Computing specializations..     Time: 0:00:02
[ Info: Computing normal forms of degree 2 in 4 variables
[ Info: Used 1 specializations in 1.2950889 seconds, found 4 relations
[ Info: Computing 5 Groebner bases for degrees (3, 3) for block orderings
✓ # Computing specializations..     Time: 0:00:03
[ Info: Computed Groebner bases in 5.368122 seconds
[ Info: Inclusion checked with probability 0.9955 in 2.4497774 seconds
[ Info: Global identifiability asses

OrderedCollections.OrderedDict{SymbolicUtils.BasicSymbolic{Real}, Symbol} with 6 entries:
  x(t) => :globally
  y(t) => :globally
  α    => :globally
  β    => :globally
  γ    => :globally
  δ    => :globally

# Setup 2. Only the sum of the states is observed


In [21]:
measured_quantities_2 = [z1 ~ x + y]  # Observing the sum of x and y
@named sys = ODESystem(eqs, t, [x, y], [α, β, γ, δ]; observed = measured_quantities_2)
sys = complete(sys)
res = assess_identifiability(sys, measured_quantities = measured_quantities_2)

┌ Info: System parsed into x' = -x*y*β + x*α
│ y' = x*y*δ - y*γ
└ z1 = x + y
[ Info: Assessing local identifiability
[ Info: Assessing global identifiability
[ Info: Functions to check involve states
[ Info: Computing IO-equations
[ Info: Computed IO-equations in 0.001462 seconds
[ Info: Computing Wronskians
[ Info: Computed Wronskians in 0.0018218 seconds
[ Info: Dimensions of the Wronskians [18]
[ Info: Ranks of the Wronskians computed in 3.67e-5 seconds
[ Info: Simplifying generating set. Simplification level: standard
[ Info: Computing normal forms of degree 2 in 4 variables
[ Info: Used 7 specializations in 0.0034154 seconds, found 8 relations
[ Info: Computing 5 Groebner bases for degrees (3, 3) for block orderings
[ Info: Computed Groebner bases in 0.123479 seconds
[ Info: Inclusion checked with probability 0.9955 in 0.0041959 seconds
[ Info: Global identifiability assessed in 0.1695562 seconds


OrderedCollections.OrderedDict{SymbolicUtils.BasicSymbolic{Real}, Symbol} with 6 entries:
  x(t) => :locally
  y(t) => :locally
  α    => :locally
  β    => :locally
  γ    => :locally
  δ    => :locally

In [ ]:
find_identifiable_functions(sys, measured_quantities = measured_quantities_2, ) # Single experiment

[ Info: Computing IO-equations
[ Info: Computed IO-equations in 0.0020338 seconds
[ Info: Computing Wronskians
[ Info: Computed Wronskians in 0.0022808 seconds
[ Info: Dimensions of the Wronskians [18]
[ Info: Ranks of the Wronskians computed in 3.42e-5 seconds
[ Info: Simplifying generating set. Simplification level: standard
[ Info: Computing normal forms of degree 2 in 4 variables
[ Info: Used 7 specializations in 0.0018559 seconds, found 8 relations
[ Info: Computing 5 Groebner bases for degrees (3, 3) for block orderings
[ Info: Computed Groebner bases in 0.0591634 seconds
[ Info: Inclusion checked with probability 0.995 in 0.0039593 seconds
[ Info: The search for identifiable functions concluded in 0.0985848 seconds


5-element Vector{Num}:
     β - δ
     α - γ
       β*δ
       α*γ
 α*δ + β*γ

## with multiple experiments
Corresponding to sample initial conditions independently for each experiment


In [ ]:
res = assess_local_identifiability(sys, measured_quantities = measured_quantities, type = :ME)

┌ Info: System parsed into x' = -x*y*β + x*α
│ y' = x*y*δ - y*γ
└ z1 = x + y
┌ Error: Multi-experiment identifiability is not properly defined for the states
└ @ StructuralIdentifiability C:\Users\MGAJ\.julia\packages\StructuralIdentifiability\Tp10P\src\local_identifiability.jl:200


ArgumentError: ArgumentError: State variable x appears in x

## with known initial conditions

In [ ]:
res = assess_identifiability(sys, measured_quantities = measured_quantities_2, known_ic = [x, y])

┌ Info: System parsed into x' = -x*y*β + x*α
│ y' = x*y*δ - y*γ
└ z1 = x + y
[ Info: Computing IO-equations
[ Info: Computed IO-equations in 0.0018903 seconds
[ Info: Computing Wronskians
[ Info: Computed Wronskians in 0.0022219 seconds
[ Info: Dimensions of the Wronskians [18]
[ Info: Ranks of the Wronskians computed in 2.93e-5 seconds
[ Info: Simplifying generating set. Simplification level: standard
[ Info: Computing normal forms of degree 2 in 4 variables
[ Info: Used 7 specializations in 0.0015091 seconds, found 8 relations
[ Info: Computing 5 Groebner bases for degrees (3, 3) for block orderings
[ Info: Computed Groebner bases in 0.0408558 seconds
[ Info: Inclusion checked with probability 0.99875 in 0.0030784 seconds
[ Info: Simplifying generating set. Simplification level: standard
[ Info: Computing normal forms of degree 2 in 6 variables
[ Info: Used 13 specializations in 0.0050461 seconds, found 15 relations
[ Info: Computing 7 Groebner bases for degrees (3, 3) for block orde

OrderedCollections.OrderedDict{SymbolicUtils.BasicSymbolic{Real}, Symbol} with 6 entries:
  x(0) => :globally
  y(0) => :globally
  α    => :globally
  β    => :globally
  γ    => :globally
  δ    => :globally